In [0]:
from pyspark.sql import functions as F

# ==========================================
# CONFIGURACIÓN
# ==========================================

TOTAL_CLIENTES = 1000
TABLA_DESTINO = "workspace.telco_bronze.clientes_raw"

# ==========================================
# GENERACIÓN DE DATOS SINTÉTICOS
# ==========================================

df_clientes = (
    spark.range(1, TOTAL_CLIENTES + 1)
    .select(
        F.concat(
            F.lit("CLI"),
            F.lpad(F.col("id").cast("string"), 6, "0")
        ).alias("cliente_id"),

        F.when(F.col("id") % 4 == 0, "Bogota")
         .when(F.col("id") % 4 == 1, "Medellin")
         .when(F.col("id") % 4 == 2, "Cali")
         .otherwise("Barranquilla")
         .alias("ciudad"),

        F.when(F.col("id") % 5 == 0, "Empresas")
         .otherwise("Personas")
         .alias("segmento"),

        F.date_format(
            F.date_add(
                F.to_date(F.lit("2023-01-01")),
                (F.col("id") % 900).cast("int")
            ),
            "yyyy-MM-dd"
        ).alias("fecha_alta_raw"),

        F.when(F.col("id") % 13 == 0, "INACTIVO")
         .otherwise("ACTIVO")
         .alias("estado_raw")
    )
    .withColumn("_fuente", F.lit("sintetico_telco_v1"))
    .withColumn("_fecha_ingesta", F.current_timestamp())
)

# ==========================================
# CREACIÓN DE TABLA DELTA
# ==========================================

if not spark.catalog.tableExists(TABLA_DESTINO):

    (
        df_clientes.write
        .format("delta")
        .mode("ignore")
        .saveAsTable(TABLA_DESTINO)
    )

    print("Tabla Bronze creada correctamente")

else:
    print("La tabla ya existe. No se modifican los datos.")

# ==========================================
# VALIDACIÓN
# ==========================================

df_resultado = spark.table(TABLA_DESTINO)

print("Total de clientes:", df_resultado.count())

display(df_resultado.limit(10))